# Módulo 38: Agrupación y particionamiento

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m38-agrupacion-y-particionamiento-solucion.md](m38-agrupacion-y-particionamiento-solucion.md). No es una tarea entregada del programa.

Una tabla de detalle de ventas puede tener millones de filas; para tomar decisiones necesitamos **resúmenes**: total por producto, promedio por orden, el artículo más caro de cada pedido. SQL ofrece dos herramientas para eso. `GROUP BY` **colapsa** las filas en un resumen por grupo, y las **funciones de ventana** calculan resúmenes **sin perder** el detalle de cada fila.

1. Funciones de agregado
2. `GROUP BY`
3. `HAVING` frente a `WHERE` y el orden lógico de ejecución
4. Funciones de ventana: `OVER` y `PARTITION BY`
5. `ORDER BY` dentro de la ventana
6. `ROW_NUMBER`, `RANK` y `DENSE_RANK`
7. Top-N por grupo
8. Errores frecuentes
9. Ejercicio

In [1]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE SalesOrderDetail (
    SalesOrderID       INTEGER,
    SalesOrderDetailID INTEGER PRIMARY KEY,
    ProductID          INTEGER,
    OrderQty           INTEGER,
    UnitPrice          REAL,
    LineTotal          REAL       -- OrderQty * UnitPrice
);
INSERT INTO SalesOrderDetail VALUES
 (43659,  1, 776, 1, 2025, 2025), (43659,  2, 777, 3, 2025, 6075),
 (43659,  3, 778, 1, 2025, 2025), (43659,  4, 771, 1, 2040, 2040),
 (43660,  5, 762, 1,  420,  420), (43660,  6, 758, 1,  875,  875),
 (43661,  7, 776, 2, 2025, 4050), (43661,  8, 747, 1,  715,  715),
 (43661,  9, 712, 4,    5,   20), (43661, 10, 715, 2,   28,   56),
 (43661, 11, 711, 2,   20,   40), (43661, 12, 716, 2,   28,   56),
 (43662, 13, 762, 3,  420, 1260), (43662, 14, 776, 1, 2025, 2025),
 (43662, 15, 758, 2,  875, 1750);
""")


def q(sql):
    """Ejecuta una consulta y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn)


q("SELECT * FROM SalesOrderDetail LIMIT 6")

,SalesOrderID,SalesOrderDetailID,ProductID,OrderQty,UnitPrice,LineTotal
0,43659,1,776,1,2025.0,2025.0
1,43659,2,777,3,2025.0,6075.0
2,43659,3,778,1,2025.0,2025.0
3,43659,4,771,1,2040.0,2040.0
4,43660,5,762,1,420.0,420.0
5,43660,6,758,1,875.0,875.0


## 1. Funciones de agregado

Una función de agregado recibe muchas filas y devuelve un solo valor. Todas ignoran los `NULL`, salvo `COUNT(*)`.

| Función | Devuelve |
| --- | --- |
| `COUNT(*)` / `COUNT(col)` | Número de filas / de valores no nulos |
| `SUM(col)` | Suma |
| `AVG(col)` | Promedio: $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$, con $n$ = valores no nulos |
| `MIN(col)`, `MAX(col)` | Mínimo y máximo |

Sin `GROUP BY`, el agregado resume la tabla entera en una fila.

In [2]:
q("""
SELECT COUNT(*)                  AS lineas,
       COUNT(DISTINCT SalesOrderID) AS ordenes,
       SUM(OrderQty)             AS unidades,
       ROUND(AVG(LineTotal), 2)  AS promedio_linea,
       MAX(LineTotal)            AS linea_maxima
FROM SalesOrderDetail
""")

,lineas,ordenes,unidades,promedio_linea,linea_maxima
0,15,4,27,1562.13,6075.0


## 2. `GROUP BY`

`GROUP BY` forma un grupo por cada valor distinto de las columnas indicadas y aplica los agregados **dentro de cada grupo**. El resultado tiene una fila por grupo.

La regla clave: **cada columna del `SELECT` debe estar en el `GROUP BY` o dentro de una función de agregado**. Si no, el motor no sabría qué valor mostrar de entre todas las filas del grupo.

In [3]:
q("""
SELECT ProductID,
       SUM(OrderQty)  AS unidades,
       SUM(LineTotal) AS ventas,
       COUNT(*)       AS lineas
FROM SalesOrderDetail
GROUP BY ProductID
ORDER BY ventas DESC
""")

,ProductID,unidades,ventas,lineas
0,776,4,8100.0,3
1,777,3,6075.0,1
2,758,3,2625.0,2
3,771,1,2040.0,1
4,778,1,2025.0,1
5,762,4,1680.0,2
6,747,1,715.0,1
7,716,2,56.0,1
8,715,2,56.0,1
9,711,2,40.0,1


## 3. `HAVING` frente a `WHERE` y el orden lógico de ejecución

Las dos cláusulas filtran, pero en momentos distintos:

- `WHERE` filtra **filas** antes de agrupar. No puede usar agregados, porque los grupos aún no existen.
- `HAVING` filtra **grupos** después de agrupar. Sí puede usar agregados.

Aunque la consulta se escribe empezando por `SELECT`, el motor la procesa en este orden lógico:

| Paso | Cláusula | Qué ocurre |
| --- | --- | --- |
| 1 | `FROM` (y `JOIN`) | Se reúnen las filas de origen |
| 2 | `WHERE` | Se descartan filas |
| 3 | `GROUP BY` | Se forman los grupos |
| 4 | `HAVING` | Se descartan grupos |
| 5 | `SELECT` | Se calculan columnas, alias y funciones de ventana |
| 6 | `ORDER BY` | Se ordena el resultado |
| 7 | `LIMIT` / `TOP` | Se recorta el número de filas |

De esta tabla salen varias reglas prácticas: un alias del `SELECT` no existe todavía en `WHERE`, y una función de ventana no se puede filtrar en `WHERE` ni en `HAVING`.

In [4]:
# Solo líneas con precio unitario mayor que 100 (WHERE),
# y solo productos que suman más de 2 unidades (HAVING)
q("""
SELECT ProductID, SUM(OrderQty) AS unidades, SUM(LineTotal) AS ventas
FROM SalesOrderDetail
WHERE UnitPrice > 100
GROUP BY ProductID
HAVING SUM(OrderQty) > 2
ORDER BY unidades DESC
""")

,ProductID,unidades,ventas
0,776,4,8100.0
1,762,4,1680.0
2,777,3,6075.0
3,758,3,2625.0


## 4. Funciones de ventana: `OVER` y `PARTITION BY`

Una **función de ventana** calcula un valor a partir de un conjunto de filas relacionadas con la fila actual (su "ventana"), pero **no colapsa** las filas: cada línea conserva su detalle y gana una columna nueva.

- `SUM(x) OVER ()` usa como ventana toda la tabla.
- `SUM(x) OVER (PARTITION BY g)` usa como ventana las filas con el mismo valor de `g`, como un `GROUP BY` que no agrupa.

Así se puede calcular, por ejemplo, qué porcentaje del total de su orden representa cada línea.

In [5]:
q("""
SELECT SalesOrderID, SalesOrderDetailID, LineTotal,
       SUM(LineTotal) OVER (PARTITION BY SalesOrderID) AS total_orden,
       ROUND(100.0 * LineTotal / SUM(LineTotal) OVER (PARTITION BY SalesOrderID), 1) AS pct_orden
FROM SalesOrderDetail
WHERE SalesOrderID IN (43659, 43660)
""")

,SalesOrderID,SalesOrderDetailID,LineTotal,total_orden,pct_orden
0,43659,1,2025.0,12165.0,16.6
1,43659,2,6075.0,12165.0,49.9
2,43659,3,2025.0,12165.0,16.6
3,43659,4,2040.0,12165.0,16.8
4,43660,5,420.0,1295.0,32.4
5,43660,6,875.0,1295.0,67.6


## 5. `ORDER BY` dentro de la ventana

Si la ventana lleva `ORDER BY`, el agregado se vuelve **acumulado**: para cada fila suma desde el inicio de la partición hasta esa fila. Es la forma de obtener totales corridos.

Ojo: el `ORDER BY` de la ventana solo define el orden del cálculo; el orden en que se muestran las filas lo decide el `ORDER BY` final de la consulta.

In [6]:
q("""
SELECT SalesOrderID, SalesOrderDetailID, LineTotal,
       SUM(LineTotal) OVER (PARTITION BY SalesOrderID ORDER BY SalesOrderDetailID) AS acumulado
FROM SalesOrderDetail
WHERE SalesOrderID IN (43659, 43660)
ORDER BY SalesOrderID, SalesOrderDetailID
""")

,SalesOrderID,SalesOrderDetailID,LineTotal,acumulado
0,43659,1,2025.0,2025.0
1,43659,2,6075.0,8100.0
2,43659,3,2025.0,10125.0
3,43659,4,2040.0,12165.0
4,43660,5,420.0,420.0
5,43660,6,875.0,1295.0


## 6. `ROW_NUMBER`, `RANK` y `DENSE_RANK`

Las tres numeran las filas de cada partición según el `ORDER BY` de la ventana. Solo se distinguen cuando hay **empates**:

| Función | Con empates | Ejemplo para valores 100, 80, 80, 50 |
| --- | --- | --- |
| `ROW_NUMBER()` | Números distintos; el desempate es arbitrario | 1, 2, 3, 4 |
| `RANK()` | Mismo número; **salta** los siguientes | 1, 2, 2, 4 |
| `DENSE_RANK()` | Mismo número; **no salta** | 1, 2, 2, 3 |

La orden 43661 tiene dos líneas de 56, así que muestra la diferencia:

In [7]:
q("""
SELECT SalesOrderDetailID, ProductID, LineTotal,
       ROW_NUMBER() OVER (ORDER BY LineTotal DESC, SalesOrderDetailID) AS row_number,
       RANK()       OVER (ORDER BY LineTotal DESC)                     AS rank,
       DENSE_RANK() OVER (ORDER BY LineTotal DESC)                     AS dense_rank
FROM SalesOrderDetail
WHERE SalesOrderID = 43661
ORDER BY LineTotal DESC, SalesOrderDetailID
""")

,SalesOrderDetailID,ProductID,LineTotal,row_number,rank,dense_rank
0,7,776,4050.0,1,1,1
1,8,747,715.0,2,2,2
2,10,715,56.0,3,3,3
3,12,716,56.0,4,3,3
4,11,711,40.0,5,5,4
5,9,712,20.0,6,6,5


A `ROW_NUMBER` se le agregó `SalesOrderDetailID` como segundo criterio para que el desempate sea siempre el mismo. Sin él, las dos líneas de 56 podrían intercambiar sus números entre ejecuciones.

## 7. Top-N por grupo

Una pregunta muy común es "las *N* filas más altas de cada grupo": los dos productos más vendidos por categoría, la línea más cara de cada orden... Como las funciones de ventana se calculan en el paso del `SELECT`, no se pueden filtrar en el `WHERE` de la misma consulta. La solución es numerar en una subconsulta (o en un CTE, como se verá en el Módulo 39) y filtrar afuera.

In [8]:
# Las 2 líneas de mayor importe de cada orden
q("""
SELECT SalesOrderID, SalesOrderDetailID, ProductID, LineTotal, posicion
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY SalesOrderID
                              ORDER BY LineTotal DESC, SalesOrderDetailID) AS posicion
    FROM SalesOrderDetail
) AS numeradas
WHERE posicion <= 2
ORDER BY SalesOrderID, posicion
""")

,SalesOrderID,SalesOrderDetailID,ProductID,LineTotal,posicion
0,43659,2,777,6075.0,1
1,43659,4,771,2040.0,2
2,43660,6,758,875.0,1
3,43660,5,762,420.0,2
4,43661,7,776,4050.0,1
5,43661,8,747,715.0,2
6,43662,14,776,2025.0,1
7,43662,15,758,1750.0,2


Si los empates deben incluirse todos (por ejemplo, dos líneas que comparten el segundo lugar), se usa `RANK` o `DENSE_RANK` en lugar de `ROW_NUMBER`.

## 8. Errores frecuentes

**Seleccionar una columna que no está agrupada ni agregada.** SQL Server rechaza la consulta. SQLite la acepta y devuelve el valor de **alguna** fila del grupo, lo que produce resultados engañosos sin aviso:

In [9]:
# SalesOrderDetailID no está en GROUP BY: SQLite elige un valor cualquiera del grupo
q("""
SELECT SalesOrderID, SalesOrderDetailID, SUM(LineTotal) AS total
FROM SalesOrderDetail
GROUP BY SalesOrderID
""")

,SalesOrderID,SalesOrderDetailID,total
0,43659,1,12165.0
1,43660,5,1295.0
2,43661,7,4937.0
3,43662,13,5035.0


En T-SQL esa consulta falla con el mensaje *"Column 'SalesOrderDetailID' is invalid in the select list because it is not contained in either an aggregate function or the GROUP BY clause"*. La forma correcta es agregar la columna (`COUNT(SalesOrderDetailID)`, `MIN(...)`) o quitarla.

**Usar un agregado en `WHERE`.** El filtro sobre un resumen va en `HAVING`:

In [10]:
try:
    q("SELECT ProductID FROM SalesOrderDetail WHERE SUM(OrderQty) > 2 GROUP BY ProductID")
except Exception as error:
    print("Error:", error)

q("SELECT ProductID, SUM(OrderQty) AS unidades FROM SalesOrderDetail GROUP BY ProductID HAVING SUM(OrderQty) > 2")

Error: Execution failed on sql 'SELECT ProductID FROM SalesOrderDetail WHERE SUM(OrderQty) > 2 GROUP BY ProductID': misuse of aggregate: SUM()


,ProductID,unidades
0,712,4
1,758,3
2,762,4
3,776,4
4,777,3


**Filtrar una función de ventana en `WHERE`.** Falla por la misma razón: la ventana todavía no existe. Se filtra en una consulta externa, como en la sección 7.

In [11]:
try:
    q("""
    SELECT SalesOrderID, LineTotal,
           ROW_NUMBER() OVER (PARTITION BY SalesOrderID ORDER BY LineTotal DESC) AS posicion
    FROM SalesOrderDetail
    WHERE posicion = 1
    """)
except Exception as error:
    print("Error:", error)

Error: Execution failed on sql '
    SELECT SalesOrderID, LineTotal,
           ROW_NUMBER() OVER (PARTITION BY SalesOrderID ORDER BY LineTotal DESC) AS posicion
    FROM SalesOrderDetail
    WHERE posicion = 1
    ': misuse of aliased window function posicion


**Confundir `PARTITION BY` con `GROUP BY`.** `GROUP BY` devuelve una fila por grupo; `PARTITION BY` devuelve todas las filas con el cálculo del grupo repetido. Si se necesita el detalle y el resumen a la vez, la herramienta es la ventana.

## 9. Ejercicio

Una cadena registra sus ventas mensuales por tienda:

```python
ventas = [  # (Tienda, Mes, Ventas)
    ("Centro", "2024-01", 120), ("Centro", "2024-02", 150), ("Centro", "2024-03", 150),
    ("Norte",  "2024-01",  90), ("Norte",  "2024-02",  70), ("Norte",  "2024-03", 110),
    ("Sur",    "2024-01",  60), ("Sur",    "2024-02",  65), ("Sur",    "2024-03",  40),
]
```

1. Calcula el total y el promedio mensual de cada tienda, y conserva solo las tiendas con un total mayor que 200.
2. Para cada fila, muestra el total de su tienda y el porcentaje que representa el mes.
3. Muestra el acumulado de ventas de cada tienda, mes a mes.
4. Ordena los meses de cada tienda de mayor a menor venta con `RANK` y `DENSE_RANK`, y obtén el mejor mes de cada tienda (si hay empate, todos los empatados).

### Solución

In [12]:
ventas = [
    ("Centro", "2024-01", 120), ("Centro", "2024-02", 150), ("Centro", "2024-03", 150),
    ("Norte",  "2024-01",  90), ("Norte",  "2024-02",  70), ("Norte",  "2024-03", 110),
    ("Sur",    "2024-01",  60), ("Sur",    "2024-02",  65), ("Sur",    "2024-03",  40),
]
conn.execute("CREATE TABLE VentaMensual (Tienda TEXT, Mes TEXT, Ventas REAL)")
conn.executemany("INSERT INTO VentaMensual VALUES (?, ?, ?)", ventas)

q("""
SELECT Tienda, SUM(Ventas) AS total, ROUND(AVG(Ventas), 1) AS promedio_mensual
FROM VentaMensual
GROUP BY Tienda
HAVING SUM(Ventas) > 200
ORDER BY total DESC
""")

,Tienda,total,promedio_mensual
0,Centro,420.0,140.0
1,Norte,270.0,90.0


In [13]:
q("""
SELECT Tienda, Mes, Ventas,
       SUM(Ventas) OVER (PARTITION BY Tienda)                 AS total_tienda,
       ROUND(100.0 * Ventas / SUM(Ventas) OVER (PARTITION BY Tienda), 1) AS pct,
       SUM(Ventas) OVER (PARTITION BY Tienda ORDER BY Mes)    AS acumulado
FROM VentaMensual
ORDER BY Tienda, Mes
""")

,Tienda,Mes,Ventas,total_tienda,pct,acumulado
0,Centro,2024-01,120.0,420.0,28.6,120.0
1,Centro,2024-02,150.0,420.0,35.7,270.0
2,Centro,2024-03,150.0,420.0,35.7,420.0
3,Norte,2024-01,90.0,270.0,33.3,90.0
4,Norte,2024-02,70.0,270.0,25.9,160.0
5,Norte,2024-03,110.0,270.0,40.7,270.0
6,Sur,2024-01,60.0,165.0,36.4,60.0
7,Sur,2024-02,65.0,165.0,39.4,125.0
8,Sur,2024-03,40.0,165.0,24.2,165.0


In [14]:
display(q("""
SELECT Tienda, Mes, Ventas,
       RANK()       OVER (PARTITION BY Tienda ORDER BY Ventas DESC) AS rank,
       DENSE_RANK() OVER (PARTITION BY Tienda ORDER BY Ventas DESC) AS dense_rank
FROM VentaMensual
ORDER BY Tienda, rank, Mes
"""))

# Mejor mes por tienda, conservando empates
q("""
SELECT Tienda, Mes, Ventas
FROM (
    SELECT *, RANK() OVER (PARTITION BY Tienda ORDER BY Ventas DESC) AS posicion
    FROM VentaMensual
) AS r
WHERE posicion = 1
ORDER BY Tienda, Mes
""")

,Tienda,Mes,Ventas,rank,dense_rank
0,Centro,2024-02,150.0,1,1
1,Centro,2024-03,150.0,1,1
2,Centro,2024-01,120.0,3,2
3,Norte,2024-03,110.0,1,1
4,Norte,2024-01,90.0,2,2
5,Norte,2024-02,70.0,3,3
6,Sur,2024-02,65.0,1,1
7,Sur,2024-01,60.0,2,2
8,Sur,2024-03,40.0,3,3


,Tienda,Mes,Ventas
0,Centro,2024-02,150.0
1,Centro,2024-03,150.0
2,Norte,2024-03,110.0
3,Sur,2024-02,65.0


Centro tiene dos meses empatados con 150, y `RANK` los conserva a ambos en el primer lugar. Con `ROW_NUMBER` solo quedaría uno.

In [15]:
conn.close()

## Resumen

- Las funciones de agregado (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) resumen muchas filas en un valor e ignoran los `NULL`.
- Con `GROUP BY`, toda columna del `SELECT` debe estar agrupada o agregada; SQLite no lo exige, pero SQL Server sí.
- `WHERE` filtra filas antes de agrupar y `HAVING` filtra grupos después; el orden lógico es FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY.
- Las funciones de ventana (`OVER`, `PARTITION BY`, `ORDER BY`) calculan resúmenes y acumulados sin colapsar filas.
- `ROW_NUMBER`, `RANK` y `DENSE_RANK` difieren en los empates; el top-N por grupo se filtra en una consulta externa.

En el **Módulo 39** se combinan subconsultas, CTE y tablas temporales para escribir consultas de varios pasos.